# P5 Vehicle Recognition: BACKUP run (faster)

The settings are already filled in for the backup run: 15 classifier epochs, 10 detector epochs, and faster ablation and stress tests on fixed test subsets. Do not change them. Use Save Version > Save & Run All.

**Before running**
1. Settings: set the accelerator to **GPU T4** and turn **Internet on** (Kaggle may ask for phone verification).
2. Add input: the Stanford Cars dataset, `rickyyyyyyy/torchvision-stanford-cars`.
3. Add input: upload `p5-vehicle-id.zip` as a new private dataset named `p5-vehicle-id`.

**Order of runs**
1. Leave `SMOKE_TEST = True` and run all cells. This tests the whole chain on a tiny model. Its numbers are not valid results.
2. Set `SMOKE_TEST = False`, fill in `TEAM_ID` and `USNS`, then use **Save Version > Save & Run All** so it keeps running with the browser closed.
3. Download `p5_outputs.zip` from the Output tab.

**If time runs out:** add this notebook's saved output as an input and run it again. Training resumes from the last saved epoch.

In [ ]:
# ===== SETTINGS: edit this cell only =====
SMOKE_TEST = False         # Backup run
TEAM_ID = "T10"
USNS = "1RUA24CSE0262,1RUA24CSE0306,1RUA24CSE0291,1RUA24CSE0300,1RUA24CSE0296"
USE_TINY_MODEL = False     # True: smaller, faster ConvNeXt V2 Tiny
EPOCHS = 15
DETECTOR_EPOCHS = 10
SESSION_HOURS = 11         # Check your own Kaggle session limit and adjust this
EVALUATION_HOURS = 4       # Evaluation is skipped if less than this much session time is left

import csv, json, os, shutil, subprocess, sys, time, zipfile
os.environ["P5_FAST_EVAL"] = "1"   # Faster ablation and stress tests
from pathlib import Path
STARTED = time.time()
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "p5-vehicle-id"
BASE_MODEL = "convnextv2_tiny.fcmae_ft_in22k_in1k_384" if USE_TINY_MODEL else "convnextv2_base.fcmae_ft_in22k_in1k_384"
CLASSIFIER_DIR = "checkpoints/smoke" if SMOKE_TEST else ("checkpoints/tiny" if USE_TINY_MODEL else "checkpoints/classifier")

def run(*command):
    """Run a command, show its output, and stop the notebook if it fails."""
    command = [str(part) for part in command]
    print("\n$", " ".join(command), flush=True)
    subprocess.run(command, check=True, cwd=REPO if REPO.exists() else WORK)

def python(*arguments):
    run(sys.executable, *arguments)

def hours_used():
    return (time.time() - STARTED) / 3600

print("Mode:", "SMOKE TEST" if SMOKE_TEST else "FULL RUN", "| Model:", BASE_MODEL)
if not SMOKE_TEST and not (TEAM_ID and USNS):
    print("WARNING: TEAM_ID or USNS is empty, so the eval report will not be generated.")

In [ ]:
# Check the GPU and Internet settings before spending time.
import urllib.request
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU found. Set the notebook accelerator to GPU T4, then run again.")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
try:
    urllib.request.urlopen("https://github.com", timeout=15)
except Exception as error:
    raise SystemExit(f"No Internet ({error}). Turn Internet on in the notebook settings.")
print("Internet is on.")

In [ ]:
# Find the project in the notebook inputs and copy it to a writable folder.
def find_project():
    scripts = sorted(INPUT.rglob("scripts/01_prepare_data.py"), key=lambda p: len(p.parts))
    if scripts:
        return scripts[0].parents[1], None
    archives = sorted(INPUT.rglob("p5-vehicle-id*.zip"))
    if archives:
        return None, archives[0]
    raise SystemExit("Project not found. Add p5-vehicle-id.zip as a notebook input.")

if not REPO.exists():
    folder, archive = find_project()
    if folder:
        shutil.copytree(folder, REPO, copy_function=shutil.copyfile)
    else:
        with zipfile.ZipFile(archive) as handle:
            handle.extractall(WORK)
print("Project ready:", REPO)
required = [REPO / "src/vehicle_id/training_utils.py", REPO / "scripts/09_recalibrate_thresholds.py"]
if not all(path.exists() for path in required) or "P5_FAST_EVAL" not in (REPO / "scripts/07_evaluate.py").read_text():
    raise SystemExit("This is an OLD copy of the project. Attach only the newest p5-vehicle-ids version, delete /kaggle/working/p5-vehicle-id, and run again.")
print("Project version check passed.")
run(sys.executable, "-m", "pip", "install", "-q", "timm>=1.0", "ultralytics", "onnx", "onnxscript", "onnxruntime-gpu", "scikit-image", "scikit-learn", "joblib", "pytest")
check = subprocess.run([sys.executable, "-c", "import onnxruntime as o; print(o.get_available_providers())"], capture_output=True, text=True)
print("ONNX Runtime providers:", check.stdout.strip() or check.stderr[-500:])
if "CUDAExecutionProvider" not in check.stdout:
    print("WARNING: ONNX Runtime cannot use the GPU, so evaluation will be slow on CPU.")

In [ ]:
# Find the dataset, build the split, and check it matches the brief.
def find_data_root():
    for meta in sorted(INPUT.rglob("cars_meta.mat"), key=lambda p: len(p.parts)):
        for folder in meta.parents:
            if folder == INPUT:
                break
            if any(folder.rglob("cars_test_annos_withlabels.mat")) and any(folder.rglob("cars_train_annos.mat")):
                return folder
    raise SystemExit("Stanford Cars files not found. Add the dataset as a notebook input and check it includes cars_test_annos_withlabels.mat.")

DATA_ROOT = find_data_root()
print("Dataset:", DATA_ROOT)
python("scripts/01_prepare_data.py", "--data-root", DATA_ROOT)
with open(REPO / "data/manifest.csv", newline="") as handle:
    rows = list(csv.DictReader(handle))
counts = {name: sum(row["split"] == name for row in rows) for name in ("train", "val", "test")}
classes = len(json.loads((REPO / "data/classes.json").read_text()))
print("Images:", counts, "| Classes:", classes)
expected = counts["train"] + counts["val"] == 8144 and counts["test"] == 8041 and classes == 196
if not expected:
    message = "Expected 8,144 training images, 8,041 test images and 196 classes."
    if SMOKE_TEST:
        print("WARNING:", message)
    else:
        raise SystemExit(message + " Use a complete dataset before the full run.")

In [ ]:
# Automated tests. They use synthetic images and take under a minute.
run(sys.executable, "-m", "pytest", "tests", "-q")

In [ ]:
# Clear old outputs when switching modes, and restore checkpoints from a previous saved run.
marker = REPO / ".run_mode"
mode = "smoke" if SMOKE_TEST else "full"
if marker.exists() and marker.read_text() != mode:
    print("Mode changed: clearing old models, results and detector checkpoints.")
    for path in [*(REPO / "models").glob("*.onnx"), REPO / "models/meta.json"]:
        path.unlink(missing_ok=True)
    shutil.rmtree(REPO / "results", ignore_errors=True)
    (REPO / "results").mkdir()
    shutil.rmtree(REPO / "checkpoints/detector", ignore_errors=True)
    shutil.rmtree(REPO / "checkpoints/smoke", ignore_errors=True)
marker.write_text(mode)

if not SMOKE_TEST and not (REPO / "checkpoints").exists():
    saved = sorted(INPUT.rglob("checkpoints/*/last.pt"))
    if saved:
        source = saved[0].parents[1]
        print("Restoring checkpoints from", source)
        shutil.copytree(source, REPO / "checkpoints", copy_function=shutil.copyfile)

In [ ]:
# Train: classical model, detector, then classifier.
if not (REPO / "models/classical.npz").exists():
    python("scripts/02_train_classical.py")

detector_weights = REPO / "checkpoints/detector/finetune/weights/best.pt"
if not detector_weights.exists():
    python("scripts/04_train_detector.py", "--epochs", 1 if SMOKE_TEST else DETECTOR_EPOCHS)

if SMOKE_TEST:
    target_epochs = 1
    python("scripts/03_train_classifier.py", "--model", "convnextv2_atto.fcmae_ft_in1k", "--epochs", 1, "--batch-size", 32, "--accumulate", 1, "--out", CLASSIFIER_DIR)
else:
    target_epochs = EPOCHS
    minutes = max(10, int((SESSION_HOURS - EVALUATION_HOURS - hours_used()) * 60))
    print(f"Classifier training may use up to {minutes} minutes in this session.")
    python("scripts/03_train_classifier.py", "--model", BASE_MODEL, "--epochs", EPOCHS, "--out", CLASSIFIER_DIR, "--max-minutes", minutes)

log = REPO / CLASSIFIER_DIR / "log.csv"
finished_epochs = len(log.read_text().splitlines()) if log.exists() else 0
TRAINING_DONE = finished_epochs >= target_epochs
print(f"Classifier epochs finished: {finished_epochs}/{target_epochs}")
if not TRAINING_DONE:
    print("Training is not finished. Save this version, add its output as an input, and run again to resume.")

In [ ]:
# Export, quantize, calibrate on validation data, then evaluate once on test data.
EVALUATED = False
if not TRAINING_DONE:
    print("Skipped: classifier training is not finished.")
elif not SMOKE_TEST and hours_used() > SESSION_HOURS - EVALUATION_HOURS:
    print("Skipped: too little session time left. Save this version, add its output as an input, and run again.")
else:
    python("scripts/05_export.py", "--checkpoint", f"{CLASSIFIER_DIR}/best.pt")
    if SMOKE_TEST:
        python("scripts/06_calibrate.py", "--gpu", "--limit", 200)
        python("scripts/07_evaluate.py", "--gpu", "--limit", 200, "--robustness-images", 20, "--deployed-images", 20)
    else:
        python("scripts/06_calibrate.py", "--gpu")
        python("scripts/07_evaluate.py", "--gpu")
    EVALUATED = True

In [ ]:
# Generate reports and show the main results.
metrics_path = REPO / "results/metrics.json"
if EVALUATED and metrics_path.exists():
    if TEAM_ID and USNS and not SMOKE_TEST:
        python("scripts/08_report.py", "--team-id", TEAM_ID, "--usns", USNS)
    results = json.loads(metrics_path.read_text())
    print("Partial smoke-test numbers: NOT valid results." if results.get("partial_run") else "Full test-set results:")
    for name, row in results["comparison"].items():
        print(f"  {name:15s} top-1 {row['top1']:.2%}  top-5 {row['top5']:.2%}")
    detection = results["detection"]
    print(f"  Detection: mean IoU {detection['mean_iou']:.3f}, AP50 {detection['ap50']:.3f}, mAP50-95 {detection['map50_95']:.3f}")
    print("  Deployed model:", results["efficiency"]["deployed_classifier"], "| CPU end-to-end mean:", f"{results['efficiency']['end_to_end_cpu']['mean_ms']:.0f} ms")
else:
    print("No new evaluation results in this run.")

In [ ]:
# Package what the app and the GitHub repository need.
output = WORK / "p5_outputs.zip"
output.unlink(missing_ok=True)
included = []
meta_path = REPO / "models/meta.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    included += [meta_path, REPO / "models/classical.npz"]
    included += [REPO / "models" / meta[key] for key in ("classifier_file", "detector_file") if key in meta]
included += [path for path in (REPO / "results").rglob("*") if path.is_file()]
included += list(REPO.glob("eval_P5_*.md"))
included += [REPO / "docs/ABLATION_AND_FAILURES.md", REPO / CLASSIFIER_DIR / "log.csv"]
with zipfile.ZipFile(output, "w", zipfile.ZIP_STORED) as archive:
    for path in included:
        if path.exists():
            archive.write(path, path.relative_to(REPO))
# Symlinks to the read-only dataset are not needed in the saved output.
shutil.rmtree(REPO / "data/yolo", ignore_errors=True)
print(f"Created {output}: {output.stat().st_size / 1e6:.1f} MB, {hours_used():.2f} h used")
print("Next:", "set SMOKE_TEST = False and Save & Run All." if SMOKE_TEST else "download p5_outputs.zip and unpack it into your repository.")